# Chunked Triton SSM Scan (check + benchmark) — T4 GPU

Validates the **chunked two-level scan** (`ChunkedSSMScanFn` in `VECTOR/triton_scan.py`) against the fused scan and a pure-PyTorch reference, then benchmarks it against the fused and old (eager exp/mul) paths.

**What to look for:**
1. `CHUNKED CHECK: PASS` — the chunked forward matches the reference and chunked grads match the fused path to `atol=1e-4`.
2. `chunked vs fused` speedup in `bench_chunked` — the two-level scan should win at large T (serial depth drops from O(T) to O(chunk + T/chunk)).

Run cells in order. Requires a **T4 GPU** runtime (Runtime → Change runtime type).

> **Failed the check?** Copy the `chunk=...` lines (fwd-vs-ref / worst grad diff) from cell 3 and paste them into the chat with the code changes.

In [ ]:
# @title 1. Environment check
import sys, os, time
import numpy as np
import torch

print(f'PyTorch {torch.__version__}, CUDA {torch.version.cuda}')
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
print(f'Python {sys.version.split()[0]}')

if torch.version.cuda is None:
    print('ERROR: No CUDA build of PyTorch. Use Runtime > Change runtime type > T4 GPU.')
    raise SystemExit(1)
if torch.cuda.device_count() == 0:
    print('ERROR: No GPU detected.')
    raise SystemExit(1)
print('Environment OK')

In [ ]:
# @title 2. Clone repo + load triton_scan (self-contained)
import sys, os

REPO_URL = 'https://github.com/nishantXnova/RETRANS-X.git'
PROJECT_DIR = '/content/RETRANS-X'

if not os.path.isdir(PROJECT_DIR):
    print('Cloning repo...')
    !git clone --quiet {REPO_URL} {PROJECT_DIR}

os.chdir(PROJECT_DIR)
print('Force-updating to origin/main...')
!git fetch --quiet origin
!git reset --hard --quiet origin/main

vec_dir = os.path.join(PROJECT_DIR, 'VECTOR')
triton_path = os.path.join(vec_dir, 'triton_scan.py')
print(f'triton_scan.py exists: {os.path.isfile(triton_path)}')
if not os.path.isfile(triton_path):
    print('STILL not found.', os.listdir(vec_dir))
    raise SystemExit(1)

# Always load the freshly pulled file; register it so later cells can
# `from triton_scan import ...` (adds vec_dir to sys.path too).
sys.path.insert(0, vec_dir)
sys.modules.pop('triton_scan', None)
import triton_scan as mod

HAS_TRITON = mod.HAS_TRITON
print(f'HAS_TRITON: {HAS_TRITON}')
if not HAS_TRITON:
    print('ERROR: Triton not available on this runtime. Expected on Colab T4 with Linux.')
    raise SystemExit(1)

In [ ]:
# @title 3. Chunked scan: correctness check
# check_fused() first (the fused path is the gradient ground truth for chunked),
# then check_chunked() compares chunked fwd vs pure-PyTorch and chunked grads vs fused.
from triton_scan import check_fused, check_chunked

ok_fused = check_fused()
print()
ok_chunked = check_chunked()
print()
print('FINAL:', 'PASS' if (ok_fused and ok_chunked) else 'FAIL')

In [ ]:
# @title 4. Chunked vs fused vs old: benchmark (fwd+bwd)
from triton_scan import bench_chunked, CHUNK_DEFAULT

print(f'CHUNK_DEFAULT = {CHUNK_DEFAULT}\n')
bench_chunked(T=4096, H=128, N=8, B=4, iters=20)
print()
bench_chunked(T=16384, H=128, N=8, B=4, iters=10)

In [ ]:
# @title 5. Chunk-size sweep (optional): which C_CHUNK is fastest at T=8192?
import torch, time
from triton_scan import ChunkedSSMScanFn

device = 'cuda'
B, T, H, N = 4, 8192, 128, 8
torch.manual_seed(0)
u = torch.randn(B, T, H, device=device)
dt = (torch.rand(B, T, H, device=device).clamp_min(1e-3) * 0.1).requires_grad_()
A = (-torch.rand(H, N, device=device).clamp_min(1e-3)).requires_grad_()
Bp = torch.randn(B, T, N, device=device).requires_grad_()
C = torch.randn(B, T, N, device=device)
D = torch.randn(H, device=device)

def run(chunk):
    h = ChunkedSSMScanFn.apply(u, dt, A, Bp, T, chunk)
    y = (h * C.unsqueeze(2)).sum(-1) + D * u
    y.pow(2).mean().backward()
    for t in (dt, A, Bp): t.grad = None

print('T=%d H=%d N=%d B=%d: fwd+bwd per C_CHUNK' % (T, H, N, B))
for chunk in (32, 64, 128, 256, 512, 1024):
    if T % chunk:
        continue
    for _ in range(3):
        run(chunk)
    torch.cuda.synchronize()
    t0 = time.perf_counter()
    for _ in range(10):
        run(chunk)
    torch.cuda.synchronize()
    ms = (time.perf_counter() - t0) / 10 * 1000
    print(f'  C_CHUNK={chunk:>4}: {ms:7.2f} ms')

In [ ]:
# @title 6. THE regime test: B=1, long T
# If chain depth is the bottleneck, chunked should win HERE (fused collapses
# to B*H=128 programs at B=1/T=65536). B=4/T<=16k was the wrong regime to bench it.
import torch, time
from triton_scan import bench_chunked, ChunkedSSMScanFn

print('=== B=1, T=32768 ===')
bench_chunked(T=32768, H=128, N=8, B=1, iters=10)
print()
print('=== B=1, T=65536 ===')
bench_chunked(T=65536, H=128, N=8, B=1, iters=5)
print()

device = 'cuda'
B, T, H, N = 1, 65536, 128, 8
torch.manual_seed(0)
u = torch.randn(B, T, H, device=device)
dt = (torch.rand(B, T, H, device=device).clamp_min(1e-3) * 0.1).requires_grad_()
A = (-torch.rand(H, N, device=device).clamp_min(1e-3)).requires_grad_()
Bp = torch.randn(B, T, N, device=device).requires_grad_()
C = torch.randn(B, T, N, device=device)
D = torch.randn(H, device=device)

def run(chunk):
    h = ChunkedSSMScanFn.apply(u, dt, A, Bp, T, chunk)
    y = (h * C.unsqueeze(2)).sum(-1) + D * u
    y.pow(2).mean().backward()
    for t in (dt, A, Bp): t.grad = None

print('T=%d H=%d N=%d B=%d: fwd+bwd per C_CHUNK' % (T, H, N, B))
for chunk in (64, 128, 256, 512, 1024, 2048):
    if T % chunk:
        continue
    for _ in range(2):
        run(chunk)
    torch.cuda.synchronize()
    t0 = time.perf_counter()
    for _ in range(5):
        run(chunk)
    torch.cuda.synchronize()
    ms = (time.perf_counter() - t0) / 5 * 1000
    print(f'  C_CHUNK={chunk:>5}: {ms:7.2f} ms')

In [ ]:
# @title 7. Final pin: B*H=192 (last unmeasured crossover point)
# RESOLVED: B*H is the driver (B=2,H=64 wins 1.18x), NOT B; no T floor in
# the winning regime (B=1 wins at T=4096/8192 too). Thresholds now
# AUTO_MAX_BH=128, AUTO_MIN_T dropped entirely. RESOLVED pin: B*H=192 LOSES
# (chunked 0.97x at B=1,H=192; 0.96x at B=3,H=64) — crossover is strictly in
# (128, 192), so 128 is the CONFIRMED ceiling. Keep 128; no further probes.
# Below is the probe that settled it (archived, re-runnable).
from triton_scan import bench_chunked

print('=== Final pin: B*H=192 — raise AUTO_MAX_BH to 192? ===')
print('  B=1, H=192, T=32768:')
bench_chunked(T=32768, H=192, N=8, B=1, iters=10)
print()
print('  B=3, H=64, T=32768 (also B*H=192):')
bench_chunked(T=32768, H=64, N=8, B=3, iters=10)

In [ ]:
# @title 8. Muon: correctness + partition sanity + stability
# Muon optimizer (Keller Jordan NS5): correctness + partition sanity + stability.
# Self-contained: loads muon.py and model.py from VECTOR.
import os, sys, importlib, math
import torch

# muon.py is UNTRACKED (not in the git clone), so bootstrap it from the
# embedded source below. Runs on EVERY execution (no 'not in globals' guard)
# and force-reimports, so code fixes actually take effect when re-run in the
# same kernel.
_MUON_SRC = r'''"""
Muon optimizer (Keller Jordan) for RETRANS-X.

Muon = Momentum orthogonalized by Newton-Schulz. It runs SGD-momentum and then
replaces each 2D update with the nearest orthogonal matrix via a quintic
Newton-Schulz iteration, computed in bfloat16 for GPU efficiency.

It applies only to 2D matmul weights (nn.Linear). Embeddings, biases, norm
weights and non-Linear parameters (e.g. A_log, D) are trained by AdamW through
the MuonAdamW hybrid wrapper.

References:
  https://kellerjordan.github.io/posts/muon/
  modded-nanogpt train_gpt2.py (pinned commit 9730304)
"""

import inspect

import torch
import torch.nn as nn


_NS_STEPS_RECT = 5      # provably robust for every non-square matrix
_NS_STEPS_SQUARE = 14   # square matrices need ~14 steps (see docstring)


@torch.no_grad()
def zeropower_via_newtonschulz5(G: torch.Tensor, steps: int = 5, eps: float = 1e-7) -> torch.Tensor:
    """Newton-Schulz quintic iteration for the zeroth power (orthogonal factor) of a 2D matrix.

    Runs in bfloat16. The matrix is transposed to rows <= cols so the polynomial
    approximation stays well-conditioned, then restored to the original orientation.

    Step count is shape-dependent (measured on this repo, 100+ random draws):
      * non-square (ratio >= 1.25, the dominant case: in_proj/out_proj are 2:1,
        x_proj 8:1, head wide): 5 steps lands singular values in ~[0.68, 1.13]
        every draw.
      * square (ratio 1.0): 5 steps can collapse the smallest singular value to
        ~0.002 (the smallest singular value of a square Gaussian tends to zero,
        and bf16 rounding amplifies it), silently distorting that gradient
        direction every step. 10 steps still lets the 512x512 (the real dt_proj
        size) tail dip to ~0.13 in worst-case bf16 and ~0.29 on real GPU draws.
        14 steps raises the worst-case floor to ~0.68 in emulation and is the
        number used here. bf16 square NS is not exact orthogonalization: the
        weakest direction can still carry ~0.2-0.5x weight on rare draws, which
        is benign for a gradient preconditioner (the guard is collapse, ~0.002).

    Stream's default config DOES hit the square case: dt_proj = Linear(hidden,
    hidden) is nn.Linear and therefore Muon-partitioned. Keep this step bump if
    model dims ever change (a future config where any two dims match re-enters
    the square path). An external caller may override steps via the parameter.

    The input scale is removed in fp32 BEFORE the bf16 cast. Quantizing first
    (G.bfloat16() then normalize) puts NS(G) and NS(3G) on different bf16 grids;
    the iteration then amplifies that mismatch to up to ~30% relative difference
    on small square matrices. Normalizing in fp32 first makes NS scale-invariant
    to ~1e-3 (bf16) instead of ~5e-2..3e-1, at no cost to the bf16 matmuls.
    """
    assert len(G.shape) == 2
    if G.size(0) == G.size(1):
        steps = max(steps, _NS_STEPS_SQUARE)
    a, b, c = (3.4445, -4.7750, 2.0315)
    X = G.float()
    X = X / (X.norm() + eps)  # remove input scale in fp32, then quantize
    X = X.bfloat16()
    if G.size(0) > G.size(1):
        X = X.T
    for _ in range(steps):
        A = X @ X.T
        B = b * A + c * A @ A
        X = a * X + B @ X
    if G.size(0) > G.size(1):
        X = X.T
    return X


class Muon(torch.optim.Optimizer):
    """Momentum orthogonalized by Newton-Schulz.

    All parameters passed in must be 2D matrices. Do not pass embeddings, the
    final projection, or any 0/1-D parameter; those go through AdamW.
    """

    def __init__(self, params, lr=0.02, momentum=0.95, nesterov=True, ns_steps=5):
        defaults = dict(lr=lr, momentum=momentum, nesterov=nesterov, ns_steps=ns_steps)
        super().__init__(params, defaults)

    def step(self, closure=None):
        loss = None
        if closure is not None:
            with torch.enable_grad():
                loss = closure()
        for group in self.param_groups:
            momentum = group['momentum']
            for p in group['params']:
                if p.grad is None:
                    continue
                g = p.grad
                state = self.state[p]
                if 'momentum_buffer' not in state:
                    state['momentum_buffer'] = torch.zeros_like(g)
                buf = state['momentum_buffer']
                buf.mul_(momentum).add_(g)
                if group['nesterov']:
                    g = g.add(buf, alpha=momentum)
                else:
                    g = buf
                g = zeropower_via_newtonschulz5(g, steps=group['ns_steps'])
                g = g * max(1, g.size(0) / g.size(1)) ** 0.5
                p.data.add_(g.to(p.data.dtype, copy=False), alpha=-group['lr'])
        return loss


def partition_params(model, include_embeddings=False):
    """Split trainable parameters into (muon_params, adamw_params).

    Muon handles nn.Linear weight matrices; everything else (embeddings, biases,
    norm weights, non-Linear params such as A_log/D) goes to AdamW.
    """
    muon = []
    seen = set()
    for _, m in model.named_modules():
        if isinstance(m, nn.Linear) and m.weight is not None and m.weight.requires_grad:
            muon.append(m.weight)
            seen.add(id(m.weight))
    if include_embeddings:
        for _, m in model.named_modules():
            if isinstance(m, nn.Embedding) and m.weight is not None and m.weight.requires_grad:
                muon.append(m.weight)
                seen.add(id(m.weight))
    adamw = [p for p in model.parameters() if p.requires_grad and id(p) not in seen]
    return muon, adamw


class MuonAdamW:
    """Hybrid optimizer: Muon for nn.Linear weights, AdamW for everything else.

    Exposes the torch.optim.Optimizer surface (param_groups, step, zero_grad,
    state_dict, load_state_dict) so it can drop into existing training loops.

    NOTE: an external LR schedule that overwrites every param_group['lr'] will
    also overwrite the Muon lr; schedule Muon and AdamW lrs explicitly.
    """

    def __init__(self, model, muon_lr=0.02, adamw_lr=6e-4, momentum=0.95, nesterov=True,
                 ns_steps=5, weight_decay=0.1, betas=(0.9, 0.95), device_type='cpu',
                 include_embeddings=False):
        muon_params, adamw_params = partition_params(model, include_embeddings=include_embeddings)
        self.muon = None
        self.adamw = None
        self.optimizers = []
        if muon_params:
            self.muon = Muon(muon_params, lr=muon_lr, momentum=momentum,
                             nesterov=nesterov, ns_steps=ns_steps)
            self.optimizers.append(self.muon)
        groups = []
        decay = [p for p in adamw_params if p.dim() >= 2]
        nodecay = [p for p in adamw_params if p.dim() < 2]
        if decay:
            groups.append({'params': decay, 'weight_decay': weight_decay})
        if nodecay:
            groups.append({'params': nodecay, 'weight_decay': 0.0})
        if groups:
            fused = ('fused' in inspect.signature(torch.optim.AdamW).parameters
                     and device_type == 'cuda')
            self.adamw = torch.optim.AdamW(groups, lr=adamw_lr, betas=betas, fused=fused)
            self.optimizers.append(self.adamw)
        if not self.optimizers:
            raise ValueError('no trainable parameters found')

    @property
    def param_groups(self):
        return [pg for opt in self.optimizers for pg in opt.param_groups]

    def step(self, closure=None):
        for opt in self.optimizers:
            opt.step(closure)

    def zero_grad(self, set_to_none=True):
        for opt in self.optimizers:
            opt.zero_grad(set_to_none=set_to_none)

    def state_dict(self):
        return [opt.state_dict() for opt in self.optimizers]

    def load_state_dict(self, state):
        for opt, s in zip(self.optimizers, state):
            opt.load_state_dict(s)
'''
_vd = None
for _p in [os.getcwd(), '/content/RETRANS-X', '/content']:
    _v = os.path.join(_p, 'VECTOR')
    if os.path.isdir(_v): _vd = _v; break
if _vd is None:
    raise RuntimeError('no VECTOR/ dir found - run CELL 1 (clone) first')
with open(os.path.join(_vd, 'muon.py'), 'w', encoding='utf-8') as _f:
    _f.write(_MUON_SRC)
sys.path.insert(0, _vd)
sys.modules.pop('muon', None)   # drop cached module, then re-import fresh
muon_mod = importlib.import_module('muon')
# hard-confirm the loaded module carries the fixes (fails loudly otherwise)
assert getattr(muon_mod, '_NS_STEPS_SQUARE', None) == 14, 'square 14-step fix missing'
assert 'fp32' in (muon_mod.zeropower_via_newtonschulz5.__doc__ or ''), 'normalize-first fix missing'
print('muon.py loaded: square 14-step + fp32-normalize-first fix active')
if 'md' not in globals():
    _vd = None
    for _p in [os.getcwd(), '/content/RETRANS-X', '/content']:
        _v = os.path.join(_p, 'VECTOR')
        if os.path.isfile(os.path.join(_v, 'model.py')): _vd = _v; break
    if _vd is None:
        for _p in [os.getcwd(), '/content/RETRANS-X', '/content']:
            if os.path.isdir(os.path.join(_p, 'VECTOR')): _vd = os.path.join(_p, 'VECTOR'); break
        if _vd is None:
            raise RuntimeError('no VECTOR/ dir found - run CELL 1 (clone) first')
    sys.path.insert(0, _vd)
    md = importlib.util.module_from_spec(
        (s := importlib.util.spec_from_file_location('md', os.path.join(_vd, 'model.py')))
    ); s.loader.exec_module(md)

# --- 1) NS: singular values in ~[0.5, 1.5], finite, scale-invariant (multi-draw) ---
# Every sub-check is folded into CELL 11 FINAL. Square 256x256 is the REAL case:
# dt_proj = Linear(hidden, hidden) is nn.Linear -> Muon-partitioned and square.
# zeropower_via_newtonschulz5 must bump square inputs to 10 steps; this test
# catches a silent collapse to sv~0 (which 5-step NS produces on square inputs).
results = []

def _band(p):
    sv = torch.linalg.svdvals(p.float()).cpu()
    return sv.min().item(), sv.max().item()

# square: 14-step bf16 NS. The band [0.2, 1.5] is deliberate: worst-case bf16
# can leave the WEAKEST direction at ~0.2-0.5x weight on rare draws (benign for
# a preconditioner); the real danger this guards against is the 5-step collapse
# to ~0.002 (direction zeroed). 0.2 sits 100x above that and below every
# measured 14-step floor (0.68 worst-case emulation at 256x256 and 512x512).
for shape, ndraws, tag, lo, hi in [((64, 64), 10, 'square 64x64', 0.2, 1.5),
                                   ((256, 256), 10, 'square 256x256', 0.2, 1.5)]:
    svmn, svmx, fin, rel = 9.0, 0.0, True, 0.0
    torch.manual_seed(0)
    for d in range(ndraws):
        G = torch.randn(*shape, device='cuda')
        P = muon_mod.zeropower_via_newtonschulz5(G)
        P2 = muon_mod.zeropower_via_newtonschulz5(3 * G)
        mn, mx = _band(P)
        svmn, svmx = min(svmn, mn), max(svmx, mx)
        fin = fin and bool(torch.isfinite(P).all())
        rel = max(rel, (P.float() - P2.float()).norm().item() / (P.float().norm().item() + 1e-9))
    ok = (lo <= svmn) and (svmx <= hi) and fin and (rel < 0.1)
    # rel floor: bf16 NS is deliberately bf16 (tensor cores); fp32 would give ~1e-6
    print(f'  {"OK " if ok else "FAIL"} NS {tag} (n={ndraws}): sv in [{svmn:.3f},{svmx:.3f}], '
          f'finite={fin}, worst scale-inv rel={rel:.2e}')
    results.append(('NS ' + tag, ok))

# rectangular: dominant case (in_proj/out_proj 2:1, x_proj 8:1, head wide); 5-step
# floor is 0.682, so the stricter [0.5, 1.5] band is honest for these.
for shape, ndraws, tag, lo, hi in [((128, 32), 10, 'rect 128x32', 0.5, 1.5),
                                   ((128, 256), 10, 'rect 128x256', 0.5, 1.5)]:
    svmn, svmx, fin, rel = 9.0, 0.0, True, 0.0
    torch.manual_seed(0)
    for d in range(ndraws):
        G = torch.randn(*shape, device='cuda')
        P = muon_mod.zeropower_via_newtonschulz5(G)
        P2 = muon_mod.zeropower_via_newtonschulz5(3 * G)
        mn, mx = _band(P)
        svmn, svmx = min(svmn, mn), max(svmx, mx)
        fin = fin and bool(torch.isfinite(P).all())
        rel = max(rel, (P.float() - P2.float()).norm().item() / (P.float().norm().item() + 1e-9))
    ok = (lo <= svmn) and (svmx <= hi) and fin and (rel < 0.1)
    print(f'  {"OK " if ok else "FAIL"} NS {tag} (n={ndraws}): sv in [{svmn:.3f},{svmx:.3f}], '
          f'finite={fin}, worst scale-inv rel={rel:.2e}')
    results.append(('NS ' + tag, ok))

# --- 2) Partition sanity on the real Stream model + shape audit ---
# Every Muon param shape is printed so a future dim change that creates a new
# square matrix (or removes the existing dt_proj square) is visible in the log.
m = md.Stream(md.StreamConfig()).cuda()
muon_params, adamw_params = muon_mod.partition_params(m)
muon_ids = {id(p) for p in muon_params}
named = dict(m.named_parameters())
in_muon = lambda name: id(named[name]) in muon_ids
checks = {
    'blocks.0.in_proj.weight in muon': in_muon('blocks.0.in_proj.weight'),
    'blocks.0.out_proj.weight in muon': in_muon('blocks.0.out_proj.weight'),
    'head.weight in muon': in_muon('head.weight'),
    'byte_embed.weight NOT in muon': not in_muon('byte_embed.weight'),
    'blocks.0.A_log NOT in muon': not in_muon('blocks.0.A_log'),
    'blocks.0.D NOT in muon': not in_muon('blocks.0.D'),
    'blocks.0.ln.weight NOT in muon': not in_muon('blocks.0.ln.weight'),
    'blocks.0.conv1d.weight NOT in muon': not in_muon('blocks.0.conv1d.weight'),
    'no param in both': len(muon_params) + len(adamw_params) == sum(
        1 for p in m.parameters() if p.requires_grad),
    'at least one square muon param (exercises square NS)': any(
        p.shape[0] == p.shape[1] for p in muon_params),
}
allok = all(checks.values())
for k, v in checks.items():
    print(f'  {"OK " if v else "FAIL"} {k}')
for p in muon_params:
    sq = ' <-- SQUARE: uses 14-step NS' if p.shape[0] == p.shape[1] else ''
    print(f'  muon shape {tuple(p.shape)}{sq}')
print(f'  muon: {len(muon_params)} params ({sum(p.numel() for p in muon_params)/1e6:.3f}M) | '
      f'adamw: {len(adamw_params)} params ({sum(p.numel() for p in adamw_params)/1e6:.3f}M)')
print('PARTITION:', 'PASS' if allok else 'FAIL')
results.append(('partition', allok))

# --- 3) Stability: 60 steps, loss must decrease, no NaN ---
torch.manual_seed(1)
m2 = md.Stream(md.StreamConfig(n_embd=64, n_layer=2, ssm_d_state=8, n_predict=2)).cuda()
opt = muon_mod.MuonAdamW(m2, muon_lr=0.02, adamw_lr=6e-4, device_type='cuda')
m2.train()
x = torch.randint(0, 256, (16, 256), device='cuda')
y = torch.randint(0, 256, (16, 256), device='cuda')
first = None
for it in range(60):
    _, loss = m2(x, targets=y)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(m2.parameters(), 1.0)
    opt.step()
    opt.zero_grad(set_to_none=True)
    if it == 0:
        first = loss.item()
last = loss.item()
stable = (last < first) and math.isfinite(last)
print(f'  {"OK " if stable else "FAIL"} stability: loss {first:.4f} -> {last:.4f}')
results.append(('stability', stable))

results.sort(key=lambda kv: kv[0])
for name, ok in results:
    print(f'  sub-check [{name}]: {"PASS" if ok else "FAIL"}')
final = all(ok for _, ok in results)
print('CELL 11 FINAL:', 'PASS' if final else 'FAIL')




In [ ]:
# @title 9. Muon vs AdamW: side-by-side (matched steps + wall-clock)
# Side-by-side: AdamW vs MuonAdamW on identical Stream models (matched steps +
# matched wall-clock). Per-step timing uses torch.cuda.synchronize() — no fake
# CPU-clock numbers. Self-contained: loads muon.py + model.py from VECTOR.
import os, sys, importlib, time, math, statistics
import torch

# muon_mod is set by CELL 11 (bootstrapped there). If this cell is run alone,
# require an existing muon.py rather than guessing.
if 'muon_mod' not in globals():
    _vd = None
    for _p in [os.getcwd(), '/content/RETRANS-X', '/content']:
        _v = os.path.join(_p, 'VECTOR')
        if os.path.isfile(os.path.join(_v, 'muon.py')): _vd = _v; break
    if _vd is None:
        raise RuntimeError('muon.py not found - run CELL 11 first (it bootstraps muon.py)')
    sys.path.insert(0, _vd)
    sys.modules.pop('muon', None)
    muon_mod = importlib.import_module('muon')
if 'md' not in globals():
    _vd = None
    for _p in [os.getcwd(), '/content/RETRANS-X', '/content']:
        _v = os.path.join(_p, 'VECTOR')
        if os.path.isfile(os.path.join(_v, 'model.py')): _vd = _v; break
    if _vd is None:
        for _p in [os.getcwd(), '/content/RETRANS-X', '/content']:
            if os.path.isdir(os.path.join(_p, 'VECTOR')): _vd = os.path.join(_p, 'VECTOR'); break
        if _vd is None:
            raise RuntimeError('no VECTOR/ dir found - run CELL 1 (clone) first')
    sys.path.insert(0, _vd)
    md = importlib.util.module_from_spec(
        (s := importlib.util.spec_from_file_location('md', os.path.join(_vd, 'model.py')))
    ); s.loader.exec_module(md)
device = 'cuda'

STEPS = 300
WARMUP = 15   # first steps excluded from timing stats (JIT/warmup/GPU clocks)
LOG = 50

def make_model():
    torch.manual_seed(0)
    return md.Stream(md.StreamConfig(
        n_embd=128, n_layer=4, ssm_d_state=8, n_predict=2,
        block_size=256, dropout=0.0, bias=False,
    )).cuda()

def run_opt(make_opt, steps, data):
    m = make_model()
    m.train()
    opt = make_opt(m)
    xs, ys = data
    losses, times = [], []
    for it in range(steps):
        torch.cuda.synchronize()
        t0 = time.perf_counter()
        _, loss = m(xs, targets=ys)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(m.parameters(), 1.0)
        opt.step()
        opt.zero_grad(set_to_none=True)
        torch.cuda.synchronize()
        times.append(time.perf_counter() - t0)
        losses.append(loss.item())
    return losses, times

torch.manual_seed(7)
data = (torch.randint(0, 256, (8, 256), device=device),
        torch.randint(0, 256, (8, 256), device=device))

print('--- AdamW (lr=6e-4, wd=0.1 on 2D, betas (0.9,0.95)) ---')
adamw_losses, adamw_times = run_opt(
    lambda m: torch.optim.AdamW(
        [{'params': [p for p in m.parameters() if p.dim() >= 2], 'weight_decay': 0.1},
         {'params': [p for p in m.parameters() if p.dim() < 2], 'weight_decay': 0.0}],
        lr=6e-4, betas=(0.9, 0.95), fused=True),
    STEPS, data)

print('--- MuonAdamW (muon_lr=0.02, adamw_lr=6e-4, wd=0.1) ---')
muon_losses, muon_times = run_opt(
    lambda m: muon_mod.MuonAdamW(m, muon_lr=0.02, adamw_lr=6e-4, device_type=device),
    STEPS, data)

def stat(times):
    # Median is the primary stat: per-step wall-clock on a shared Colab GPU is
    # skewed by scheduler contention, so mean is only reported alongside it.
    ts_ = sorted(times[WARMUP:])
    n = len(ts_)
    med = ts_[n // 2] if n % 2 else (ts_[n // 2 - 1] + ts_[n // 2]) / 2
    mean = sum(ts_) / n
    k = max(1, int(0.1 * n))
    trim = ts_[k:-k] if n - 2 * k >= 2 else ts_
    tmean = sum(trim) / len(trim)
    p90, p10 = ts_[min(n - 1, int(0.9 * n))], ts_[min(n - 1, int(0.1 * n))]
    return med, mean, tmean, p90 - p10, n

a_med, a_mean, a_tmean, a_spread, n_used = stat(adamw_times)
m_med, m_mean, m_tmean, m_spread, _ = stat(muon_times)
print(f'per-step [{n_used} used, warmup {WARMUP} dropped]')
print(f'  AdamW: median {a_med*1000:6.1f} ms | mean {a_mean*1000:6.1f} | trimmed {a_tmean*1000:6.1f} | p90-p10 {a_spread*1000:5.1f}')
print(f'  Muon : median {m_med*1000:6.1f} ms | mean {m_mean*1000:6.1f} | trimmed {m_tmean*1000:6.1f} | p90-p10 {m_spread*1000:5.1f}')
print(f'  Muon/AdamW per-step overhead (median): {m_med/a_med:.2f}x | (mean): {m_mean/a_mean:.2f}x')

print('\nloss curves (matched steps):')
for i in range(0, STEPS, LOG):
    print(f'  step {i+1:4d}: AdamW {adamw_losses[i]:.4f} | Muon {muon_losses[i]:.4f}')

# matched wall-clock (median-based): how many Muon steps fit into STEPS AdamW steps?
budget_steps = min(STEPS, int(STEPS * a_med / m_med))
print(f'\nAt matched wall-clock ({STEPS} AdamW steps @ median), Muon gets ~{budget_steps} steps')
print(f'  AdamW loss @ step {STEPS}: {adamw_losses[-1]:.4f}')
print(f'  Muon   loss @ step {budget_steps}: {muon_losses[budget_steps-1]:.4f}')

ok_adamw = math.isfinite(adamw_losses[-1])
ok_muon = math.isfinite(muon_losses[-1])
# also require the timing spread to be reported loudly, not silently averaged away
spread_note = ' (WARN: median/mean differ >15% - scheduler noise, quote MEDIAN)' \
    if abs(m_mean - m_med) / max(m_med, 1e-9) > 0.15 else ''
print(f'Timing spread check: mean/median Muon delta {abs(m_mean-m_med)/max(m_med,1e-9)*100:.1f}%{spread_note}')
print('\nVERDICT (sanity):', 'PASS' if (ok_adamw and ok_muon) else 'FAIL')
print('Paste the loss curves + per-step median/mean into the Muon validation note.')


## Retrieval A/B: does a sparse-retrieval layer close the GPT gap?\n\n**Why:** Stream is O(n) but GPT beats it on loss at matched small scale (~0.49 nat gap) — a fixed d_state=16 recurrence cannot do content-based recall like attention. This cell tests lever 1: add one sparse-retrieval block (causal sliding-window attention + global tokens, relative-position bias, O(n) memory).\n\n**Self-contained:** the `model.py` source is embedded and always rewritten, so this cell works even if the Colab clone is stale (no dependency on `git pull`). Progress is logged every ~100 steps so it never looks frozen.\n\n**Models (matched scale, identical data/steps):**\n- `Stream-4L` — 4 SSM blocks, 128D\n- `StreamR-4L` — 3 SSM + 1 retrieval block (window=128, 16 global tokens), 128D\n- `GPT-3L` — Transformer w/ RoPE, byte-level, 128D (the reference competitor)\n\n**Legs:** T=4096 (quality) and T=16384 (long context: val loss + step-time). GPT is O(n^2) so its 16K leg is the slow part — that is the point.\n\nPaste results (final val losses + per-step medians) into index.html §5.11.

In [ ]:
# @title 10. Retrieval A/B: StreamR vs Stream vs GPT (T=4096 & T=16384)# A/B: does a sparse-retrieval layer close the GPT gap?# StreamR (3 SSM + 1 retrieval) vs Stream (4 SSM) vs GPT (3L, RoPE) - same bytes,# same batches, same steps, matched scale. Legs: T=4096 (quality) and T=16384# (long context: val loss + step time). All three models see IDENTICAL data.# Caveat: the GPT T=16384 leg is O(n^2) attention and will be the slow part -# that is exactly the point (Stream/StreamR are O(n)).# SELF-CONTAINED: model.py source is embedded below and ALWAYS rewritten, so this# cell runs even if the Colab clone is stale - no dependency on git pull.import os, sys, time, math, importlibimport numpy as npimport torchimport torch.nn as nnimport torch.nn.functional as F_MODEL_SRC = r'''"""Stream: Continuous Byte-Level SSMToken-free, position-free, O(n) language model.Predicts next N bytes directly from raw bytes — no tokenizer, no PE, no gate, no MoE.Architecture:- Byte embedding (256 → D) — the only "vocabulary"- Stacked SSM blocks — recurrence = position by construction- Optional sparse-retrieval blocks (windowed attention + global tokens) for  content-based recall that a fixed d_state recurrence cannot do, while staying  O(n) memory (window, not full attention). Off by default (n_retrieval=0).- Multi-byte head: predict next N bytes per position- Single loss: next-byte CE summed over N future predictions"""import mathimport torchimport torch.nn as nnimport torch.nn.functional as Ffrom dataclasses import dataclassfrom typing import Optional, Tuple, List, TYPE_CHECKING# -----------------------------------------------------------------------------# SSM scan: JIT-compiled sequential recurrence.# On CPU the sequential loop is optimal (Blelloch tree scan adds overhead from# non-contiguous access). JIT eliminates Python loop overhead.# -----------------------------------------------------------------------------# ── JIT-compiled forward/backward scan loops ───────────────────────# TorchScript fuses the per-step elementwise ops into a single CUDA kernel,# eliminating the O(T) kernel-launch overhead from the Python loop.@torch.jit.scriptdef _ssm_fwd(a_vec: torch.Tensor, b_vec: torch.Tensor, T_s: int) -> torch.Tensor:    Bs, _, Hc, Nc = a_vec.shape    h = torch.zeros(Bs, Hc, Nc, device=a_vec.device)    out = torch.empty(Bs, T_s, Hc, Nc, device=a_vec.device)    for t in range(T_s):        h = h * a_vec[:, t] + b_vec[:, t]        out[:, t] = h    return out@torch.jit.scriptdef _ssm_bwd(grad_output: torch.Tensor, a_vec: torch.Tensor,             out: torch.Tensor) -> List[torch.Tensor]:    Bs, T_s, Hc, Nc = a_vec.shape    grad_a = torch.zeros_like(a_vec); grad_b = torch.zeros_like(a_vec)    dh = torch.zeros(Bs, Hc, Nc, device=a_vec.device)    for t in range(T_s - 1, -1, -1):        dh_total = grad_output[:, t] + dh        h_prev = out[:, t - 1] if t > 0 else torch.zeros(Bs, Hc, Nc, device=a_vec.device)        grad_b[:, t] = dh_total; grad_a[:, t] = dh_total * h_prev        dh = dh_total * a_vec[:, t]    return [grad_a, grad_b]class SSMScanFn(torch.autograd.Function):    """    Custom autograd Function wrapping JIT-compiled scan kernels.    The JIT-compiled forward/backward loops are fused into single CUDA    kernels, eliminating per-step Python overhead and most kernel-launch    overhead. The custom backward avoids building the full O(T) autograd    graph that PyTorch would construct from the loop.    """    @staticmethod    def forward(ctx, a_vec, b_vec, T_s):        out = _ssm_fwd(a_vec, b_vec, T_s)        ctx.save_for_backward(a_vec, out)        ctx.T_s = T_s        return out    @staticmethod    def backward(ctx, grad_output):        a_vec, out = ctx.saved_tensors        grad_a, grad_b = _ssm_bwd(grad_output, a_vec, out)        return grad_a, grad_b, Nonedef _ssm_scan(a_vec, b_vec, T):    """Wrapper that calls SSMScanFn.apply."""    return SSMScanFn.apply(a_vec, b_vec, T)def parallel_ssm_scan(u: torch.Tensor, dt: torch.Tensor,                      A: torch.Tensor, B: torch.Tensor, C: torch.Tensor,                      D: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:    """    SSM scan: sequential recurrence h_{t+1} = a_t · h_t + b_t, h_0 = 0,    with a_t and b_t being functions of the input.    Uses a JIT-compiled loop over T to eliminate Python overhead.    On CPU there is no O(log T) parallel advantage (tree scan adds non-contiguous    access cost), but the JIT avoids O(T) Python-level iteration cost.    Args:      u:  (B, T, H)    input      dt: (B, T, H)    step sizes      A:  (H, N)       state matrix (negative = -exp(A_log))      B:  (B, T, N)    input projection      C:  (B, T, N)    output projection      D:  (H,)         skip connection    Returns:      y:     (B, T, H)  output      state: (B, H, N)  final hidden state (detached)    """    Bs, T, H = u.shape    N = A.shape[-1]    # Precompute transition a_t and input b_t    a_vec = torch.exp(dt.unsqueeze(-1) * A.unsqueeze(0).unsqueeze(0))  # (B, T, H, N)    b_vec = dt.unsqueeze(-1) * B.unsqueeze(2) * u.unsqueeze(-1)        # (B, T, H, N)    # Custom autograd scan — h[t] = state after processing input t    h = _ssm_scan(a_vec, b_vec, T)  # (B, T, H, N)    # Output: y[t] = (h[t] · C[t]).sum(-1) + D · u[t]    y = (h * C.unsqueeze(2)).sum(-1) + D * u    return y, (h[:, -1].detach(), None)# -----------------------------------------------------------------------------# SSM Block: selective state space (Mamba-style)# -----------------------------------------------------------------------------class SSMBlock(nn.Module):    def __init__(self, n_embd: int, ssm_d_state: int = 16,                 ssm_d_conv: int = 4, ssm_expand: int = 2, bias: bool = False):        super().__init__()        self.n_embd = n_embd        self.ssm_d_state = ssm_d_state        self.ssm_d_conv = ssm_d_conv        hidden = n_embd * ssm_expand        self.in_proj = nn.Linear(n_embd, hidden * 2, bias=bias)        self.conv1d = nn.Conv1d(hidden, hidden, kernel_size=ssm_d_conv,                                padding=ssm_d_conv - 1, groups=hidden, bias=bias)        self.act = nn.SiLU()        self.x_proj = nn.Linear(hidden, ssm_d_state * 2, bias=bias)        self.dt_proj = nn.Linear(hidden, hidden, bias=True)        self.A_log = nn.Parameter(torch.zeros(hidden, ssm_d_state))        self.D = nn.Parameter(torch.randn(hidden))        self.out_proj = nn.Linear(hidden, n_embd, bias=bias)        self.ln = nn.LayerNorm(n_embd)    def forward(self, x: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:        B, T, D = x.shape        H = self.n_embd * (D // self.n_embd) if D != self.n_embd else self.n_embd * 2        x_proj = self.in_proj(x)        x_main, gate = x_proj.chunk(2, dim=-1)        x_main = self.act(x_main)        gate = torch.sigmoid(gate)        x_conv = self.conv1d(x_main.transpose(1, 2))[..., :T].transpose(1, 2)        x_conv = self.act(x_conv)        dt = F.softplus(self.dt_proj(x_conv))        B_param, C_param = self.x_proj(x_conv).chunk(2, dim=-1)        A = -torch.exp(self.A_log.float())        y, state = self._ssm_scan(x_conv, dt, A, B_param, C_param)        y = y * gate        out = self.out_proj(y)        return self.ln(out + x), state    def _ssm_scan(self, u, dt, A, B, C):        return parallel_ssm_scan(u, dt, A, B, C, self.D)# -----------------------------------------------------------------------------# Sparse Retrieval Block: causal sliding-window attention + global tokens# -----------------------------------------------------------------------------class RetrievalBlock(nn.Module):    """    Gives the SSM backbone content-based recall which a fixed d_state recurrence    cannot do, while keeping O(n) memory: each query attends to the last `window`    positions (sliding window) plus a small set of global key/value tokens.    - Window keys come from a left-padded unfold, so slice t covers positions      [t-window+1, t] — causal by construction, no triangular mask needed.    - Relative-position bias (translation-invariant) instead of absolute PE, so      the model stays position-free like the SSM recurrence it sits on.    - Global tokens are static learned memory in this v1 (sink-token style);      content-derived global keys/values are the natural v2 upgrade.    Output: post-norm residual like SSMBlock (ln(proj(y) + x), None).    """    def __init__(self, n_embd: int, n_head: int = 4, window: int = 128,                 n_global: int = 16, bias: bool = False):        super().__init__()        assert n_embd % n_head == 0        self.n_embd = n_embd        self.n_head = n_head        self.window = window        self.n_global = n_global        self.head_dim = n_embd // n_head        self.qkv = nn.Linear(n_embd, 3 * n_embd, bias=bias)        self.proj = nn.Linear(n_embd, n_embd, bias=bias)        self.g_k = nn.Parameter(torch.randn(n_global, n_head, self.head_dim) * 0.02)        self.g_v = nn.Parameter(torch.randn(n_global, n_head, self.head_dim) * 0.02)        self.rel_bias = nn.Parameter(torch.zeros(2 * window - 1))        self.ln = nn.LayerNorm(n_embd)    def forward(self, x: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:        B, T, D = x.shape        nh, hd, w, ng = self.n_head, self.head_dim, self.window, self.n_global        q, k, v = self.qkv(x).chunk(3, dim=-1)            # (B, T, D)        q = q.view(B, T, nh, hd).transpose(1, 2)          # (B, nh, T, hd)        k_pad = F.pad(k, (0, 0, w - 1, 0))        v_pad = F.pad(v, (0, 0, w - 1, 0))        k_win = k_pad.unfold(1, w, 1).view(B, T, nh, hd, w).transpose(1, 2)  # (B, nh, T, hd, w)        v_win = v_pad.unfold(1, w, 1).view(B, T, nh, hd, w).transpose(1, 2)        lw = torch.einsum('bhtd,bhtdw->bhtw', q, k_win) * (hd ** -0.5)       # (B, nh, T, w)        dist = (w - 1) - torch.arange(w, device=x.device)        lw = lw + self.rel_bias[dist].view(1, 1, 1, w)        lg = torch.einsum('bhtd,ghd->bhtg', q, self.g_k) * (hd ** -0.5)      # (B, nh, T, ng)        att = torch.softmax(torch.cat([lg, lw], dim=-1), dim=-1)             # (B, nh, T, ng+w)        ow = torch.einsum('bhtw,bhtdw->bhtd', att[..., ng:], v_win)        og = torch.einsum('bhtg,ghd->bhtd', att[..., :ng], self.g_v)        y = (ow + og).transpose(1, 2).reshape(B, T, D)        return self.ln(self.proj(y) + x), None# -----------------------------------------------------------------------------# Stream Model# -----------------------------------------------------------------------------@dataclassclass StreamConfig:    vocab_size: int = 256    n_embd: int = 256    n_layer: int = 6    ssm_d_state: int = 16    n_predict: int = 4    block_size: int = 1024    dropout: float = 0.0    bias: bool = False    # Sparse retrieval (off by default). Retrieval blocks replace the LAST    # n_retrieval SSM blocks so the head directly sees retrieved content.    n_retrieval: int = 0    n_attn_head: int = 4    window_size: int = 128    n_global: int = 16class Stream(nn.Module):    def __init__(self, config: StreamConfig):        super().__init__()        self.config = config        self.byte_embed = nn.Embedding(config.vocab_size, config.n_embd)        n_ssm = max(0, config.n_layer - config.n_retrieval)        self.blocks = nn.ModuleList(            [SSMBlock(config.n_embd, ssm_d_state=config.ssm_d_state, bias=config.bias)             for _ in range(n_ssm)]            + [RetrievalBlock(config.n_embd, n_head=config.n_attn_head,                              window=config.window_size, n_global=config.n_global,                              bias=config.bias)               for _ in range(config.n_retrieval)]        )        self.ln_f = nn.LayerNorm(config.n_embd)        self.head = nn.Linear(            config.n_embd,            config.n_predict * config.vocab_size,            bias=False        )        self.apply(self._init_weights)        for pn, p in self.named_parameters():            # out_proj / c_proj / retrieval proj get GPT-2-style residual scaling.            # '.proj.weight' matches only the retrieval block's proj (dt_proj and            # out_proj have an underscore before 'proj', so they don't match).            if (pn.endswith('out_proj.weight') or pn.endswith('c_proj.weight')                    or pn.endswith('.proj.weight')):                nn.init.normal_(p, mean=0.0, std=0.02 / math.sqrt(2 * config.n_layer))        print(f"Stream parameters: {self.get_num_params() / 1e6:.2f}M")    def get_num_params(self):        return sum(p.numel() for p in self.parameters())    def _init_weights(self, module):        if isinstance(module, nn.Linear):            nn.init.normal_(module.weight, mean=0.0, std=0.02)            if module.bias is not None:                nn.init.zeros_(module.bias)        elif isinstance(module, nn.Embedding):            nn.init.normal_(module.weight, mean=0.0, std=0.02)    def forward(self, idx: torch.Tensor,                targets: Optional[torch.Tensor] = None,                return_logits: bool = False,                iter_num: int = 0):        B, T = idx.shape        assert T <= self.config.block_size        x = self.byte_embed(idx)        for block in self.blocks:            x, _ = block(x)        x = self.ln_f(x)        logits = self.head(x)        if targets is not None:            loss = self._compute_loss(logits, targets)        else:            loss = None        if return_logits:            return logits, loss        return logits, loss    def _compute_loss(self, logits, targets):        B, T, _ = logits.shape        np = self.config.n_predict        vs = self.config.vocab_size        logits = logits.view(B, T, np, vs)        loss = 0.0        for k in range(np):            loss = loss + F.cross_entropy(                logits[:, :T - k, k].reshape(-1, vs),                targets[:, k:].reshape(-1),                ignore_index=-1            )        return loss / np    def configure_optimizers(self, weight_decay, learning_rate, betas, device_type):        import inspect        param_dict = {pn: p for pn, p in self.named_parameters() if p.requires_grad}        decay_params = [p for n, p in param_dict.items() if p.dim() >= 2]        nodecay_params = [p for n, p in param_dict.items() if p.dim() < 2]        optim_groups = [            {'params': decay_params, 'weight_decay': weight_decay},            {'params': nodecay_params, 'weight_decay': 0.0}        ]        fused_available = 'fused' in inspect.signature(torch.optim.AdamW).parameters        use_fused = fused_available and device_type == 'cuda'        extra_args = dict(fused=True) if use_fused else dict()        optimizer = torch.optim.AdamW(optim_groups, lr=learning_rate, betas=betas, **extra_args)        print(f"using fused AdamW: {use_fused}")        return optimizer    @torch.no_grad()    def generate(self, idx, max_new_tokens, temperature=1.0, top_k=None):        self.eval()        np = self.config.n_predict        vs = self.config.vocab_size        generated = 0        while generated < max_new_tokens:            idx_cond = idx[:, -self.config.block_size:]            logits, _ = self(idx_cond)            logits = logits[:, -1, :].view(1, np, vs) / temperature            for k in range(np):                if generated >= max_new_tokens:                    break                probs = F.softmax(logits[:, k], dim=-1)                if top_k is not None:                    v, _ = torch.topk(probs, top_k)                    probs[probs < v[:, [-1]]] = 0.0                    probs = probs / probs.sum(dim=-1, keepdim=True)                idx_next = torch.multinomial(probs, num_samples=1)                idx = torch.cat((idx, idx_next), dim=1)                generated += 1        self.train()        return idx'''# --- bootstrap: write model.py fresh, then import it (drop cached module) ---_vd = Nonefor _p in [os.getcwd(), '/content/RETRANS-X', '/content']:    _v = os.path.join(_p, 'VECTOR')    if os.path.isdir(_v): _vd = _v; breakif _vd is None:    raise RuntimeError('no VECTOR/ dir found - run CELL 1 (clone) first')with open(os.path.join(_vd, 'model.py'), 'w', encoding='utf-8') as _f:    _f.write(_MODEL_SRC)sys.path.insert(0, _vd)sys.modules.pop('md', None)md = importlib.util.module_from_spec(    (s := importlib.util.spec_from_file_location('md', os.path.join(_vd, 'model.py')))); s.loader.exec_module(md)assert 'n_retrieval' in md.StreamConfig.__dataclass_fields__, 'embedded model.py missing n_retrieval'print('model.py loaded (embedded, always-fresh): retrieval support OK')device = 'cuda'# --- The free efficiency wins: Triton auto-scan + fp16 autocast ---USE_FP16 = True   # fp16 autocast + GradScaler; set False if any NaN shows upUSE_TRITON = True # auto path (fused/chunked) proven on T4; set False to use JITtry:    import triton_scan as _ts    _HAS_TS = getattr(_ts, 'HAS_TRITON', False)except Exception:    _HAS_TS = Falseprint(f'efficiency: USE_FP16={USE_FP16} USE_TRITON={USE_TRITON} HAS_TRITON={_HAS_TS}')if USE_TRITON and not _HAS_TS:    print('WARNING: Triton unavailable - falling back to JIT scan (slower)')if USE_TRITON and _HAS_TS:    # local triton_scan.py may be stale (a fresh clone is needed for the fp16    # backward-kernel fix). Detect it; otherwise fp16 crashes the old kernels.    _tsrc = open(_ts.__file__, encoding='utf-8').read()    if 'h_prev dtype = out dtype' not in _tsrc:        print('!!! local triton_scan.py is STALE (missing fp16 backward fix).', flush=True)        print('    Re-run CELL 1 (git reset to origin/main) to refresh it.', flush=True)        print('    For this run: falling back to the JIT scan so training still works.', flush=True)        USE_TRITON = Falsetorch.backends.cudnn.benchmark = True# --- data: reuse VECTOR/data/bytes if present, else download+prepare TinyStories ---DATA_DIR = os.path.join(_vd, 'data', 'bytes')TRAIN_BIN = os.path.join(DATA_DIR, 'train.bin')VAL_BIN = os.path.join(DATA_DIR, 'val.bin')if not (os.path.isfile(TRAIN_BIN) and os.path.isfile(VAL_BIN)):    print('byte dataset missing - downloading TinyStories + preparing (one-time)...')    import requests    os.makedirs(DATA_DIR, exist_ok=True)    src = os.path.join(DATA_DIR, 'TinyStories-train.txt')    if not os.path.isfile(src):        url = 'https://huggingface.co/datasets/roneneldan/TinyStories/resolve/main/TinyStories-train.txt'        print('Downloading ~940MB TinyStories-train.txt...')        r = requests.get(url, stream=True)        with open(src, 'wb') as f:            for chunk in r.iter_content(chunk_size=1 << 16):                f.write(chunk)    with open(src, 'rb') as f:        raw = f.read()    split = int(len(raw) * 0.99)    np.frombuffer(raw[:split], dtype=np.uint8).copy().tofile(TRAIN_BIN)    np.frombuffer(raw[split:], dtype=np.uint8).copy().tofile(VAL_BIN)    print(f'train {split:,}B / val {len(raw)-split:,}B')train_data = np.memmap(TRAIN_BIN, dtype=np.uint8, mode='r')val_data = np.memmap(VAL_BIN, dtype=np.uint8, mode='r')# --- GPT baseline: byte-level transformer with RoPE (position-free, no wpe growth) ---def _rope(positions, dim, base=10000.0):    inv = 1.0 / (base ** (torch.arange(0, dim, 2, device=positions.device).float() / dim))    ang = positions.float().unsqueeze(-1) * inv.unsqueeze(0)   # (T, dim/2)    return torch.cos(ang), torch.sin(ang)def _rot(x):    a, b = x.chunk(2, dim=-1)    return torch.cat((-b, a), dim=-1)class GPTAttn(nn.Module):    def __init__(self, n_embd, n_head, bias):        super().__init__()        self.n_head = n_head        self.c_attn = nn.Linear(n_embd, 3 * n_embd, bias=bias)        self.c_proj = nn.Linear(n_embd, n_embd, bias=bias)    def forward(self, x, pos):        B, T, C = x.shape        q, k, v = self.c_attn(x).split(C, dim=-1)        hd = C // self.n_head        cos, sin = _rope(pos, hd)   # angles over the head dim (hd), not full n_embd        q = q.view(B, T, self.n_head, hd).transpose(1, 2)        k = k.view(B, T, self.n_head, hd).transpose(1, 2)        v = v.view(B, T, self.n_head, hd).transpose(1, 2)        q = q * cos.unsqueeze(0).unsqueeze(0) + _rot(q) * sin.unsqueeze(0).unsqueeze(0)        k = k * cos.unsqueeze(0).unsqueeze(0) + _rot(k) * sin.unsqueeze(0).unsqueeze(0)        y = F.scaled_dot_product_attention(q, k, v, is_causal=True)        return self.c_proj(y.transpose(1, 2).contiguous().view(B, T, C))class GPTMLP(nn.Module):    def __init__(self, n_embd, bias):        super().__init__()        self.c_fc = nn.Linear(n_embd, 4 * n_embd, bias=bias)        self.c_proj = nn.Linear(4 * n_embd, n_embd, bias=bias)    def forward(self, x):        return self.c_proj(F.gelu(self.c_fc(x)))class GPTBlock(nn.Module):    def __init__(self, n_embd, n_head, bias):        super().__init__()        self.ln1 = nn.LayerNorm(n_embd, bias=bias)        self.attn = GPTAttn(n_embd, n_head, bias)        self.ln2 = nn.LayerNorm(n_embd, bias=bias)        self.mlp = GPTMLP(n_embd, bias)class GPTByte(nn.Module):    def __init__(self, n_embd=128, n_layer=3, n_head=4, n_predict=4, vocab_size=256, bias=False):        super().__init__()        self.n_predict = n_predict        self.wte = nn.Embedding(vocab_size, n_embd)        self.blocks = nn.ModuleList([GPTBlock(n_embd, n_head, bias) for _ in range(n_layer)])        self.ln_f = nn.LayerNorm(n_embd, bias=bias)        self.head = nn.Linear(n_embd, n_predict * vocab_size, bias=False)        self.apply(self._init)        for pn, p in self.named_parameters():            if pn.endswith('c_proj.weight'):                nn.init.normal_(p, mean=0.0, std=0.02 / math.sqrt(2 * n_layer))    def _init(self, m):        if isinstance(m, nn.Linear):            nn.init.normal_(m.weight, mean=0.0, std=0.02)            if m.bias is not None: nn.init.zeros_(m.bias)        elif isinstance(m, nn.Embedding):            nn.init.normal_(m.weight, mean=0.0, std=0.02)    def forward(self, idx, targets=None):        B, T = idx.shape        x = self.wte(idx)        pos = torch.arange(T, device=idx.device)        for blk in self.blocks:            x = x + blk.attn(blk.ln1(x), pos)            x = x + blk.mlp(blk.ln2(x))        x = self.ln_f(x)        logits = self.head(x)        loss = None        if targets is not None:            np_, vs = self.n_predict, logits.shape[-1] // self.n_predict            lg = logits.view(B, T, np_, vs)            loss = sum(F.cross_entropy(lg[:, :T-k, k].reshape(-1, vs),                                       targets[:, k:].reshape(-1), ignore_index=-1)                       for k in range(np_)) / np_        return logits, loss# --- shared data + eval ---def make_batches(data, T, B, n, gen):    L = len(data)    out = []    for _ in range(n):        ix = torch.randint(L - T - 1, (B,), generator=gen)        x = torch.stack([torch.from_numpy(data[i:i+T].astype(np.int64)) for i in ix]).to(device)        y = torch.stack([torch.from_numpy(data[i+1:i+1+T].astype(np.int64)) for i in ix]).to(device)        out.append((x, y))    return outdef eval_loss(model, batches):    model.eval()    tot, n = 0.0, 0    with torch.no_grad():        if USE_FP16:            with torch.autocast(device_type='cuda', dtype=torch.float16):                for x, y in batches:                    _, loss = model(x, targets=y)                    tot += loss.item(); n += 1        else:            for x, y in batches:                _, loss = model(x, targets=y)                tot += loss.item(); n += 1    model.train()    return tot / ndef med(ts, drop=5):    t = sorted(ts[drop:])    n = len(t)    return t[n // 2] if n % 2 else (t[n // 2 - 1] + t[n // 2]) / 2def train_model(name, model, train_batches, val_batches, steps, warmup, eval_at,                peak_lr=6e-4, min_lr=6e-5):    model.train()    decay = [p for p in model.parameters() if p.dim() >= 2]    nodecay = [p for p in model.parameters() if p.dim() < 2]    opt = torch.optim.AdamW(        [{'params': decay, 'weight_decay': 0.1},         {'params': nodecay, 'weight_decay': 0.0}],        lr=peak_lr, betas=(0.9, 0.95), fused=True)    log_interval = max(10, steps // 12)    scaler = torch.cuda.amp.GradScaler(enabled=USE_FP16)    times, evals = [], []    t_start = time.perf_counter()    print(f'  training {name}: {steps} steps (progress every {log_interval})...', flush=True)    for it in range(steps):        if it < warmup:            lr = peak_lr * (it + 1) / warmup        else:            r = (it - warmup) / max(1, steps - warmup)            lr = min_lr + 0.5 * (peak_lr - min_lr) * (1 + math.cos(math.pi * r))        for pg in opt.param_groups: pg['lr'] = lr        x, y = train_batches[it]        torch.cuda.synchronize(); t0 = time.perf_counter()        if USE_FP16:            with torch.autocast(device_type='cuda', dtype=torch.float16):                _, loss = model(x, targets=y)            scaler.scale(loss).backward()        else:            _, loss = model(x, targets=y)            loss.backward()        if USE_FP16:            scaler.unscale_(opt)            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)            scaler.step(opt)            scaler.update()        else:            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)            opt.step()        opt.zero_grad(set_to_none=True)        torch.cuda.synchronize(); times.append(time.perf_counter() - t0)        if (it + 1) % log_interval == 0:            el = time.perf_counter() - t_start            print(f'    {name}: step {it+1}/{steps} loss {loss.item():.4f} lr {lr:.1e} ({el:.0f}s elapsed)', flush=True)        if it in eval_at:            evals.append((it + 1, eval_loss(model, val_batches)))    return evals, timesdef run_leg(T, steps, warmup, B, tag, skip=()):    print(f'\n{"="*70}', flush=True)    print(f'LEG {tag}: T={T}, B={B}, steps={steps}  (identical data for all models)', flush=True)    print(f'{"="*70}', flush=True)    tr = make_batches(train_data, T, B, steps, torch.Generator().manual_seed(2026))    va = make_batches(val_data, T, B, 60, torch.Generator().manual_seed(1))    eval_at = set([steps // 4, steps // 2, steps - 1])    models = {}    builders = {        'Stream-10M': lambda: md.Stream(md.StreamConfig(            n_embd=256, n_layer=16, n_retrieval=0, ssm_d_state=16, n_predict=4,            block_size=T, dropout=0.0, bias=False)).to(device),        'StreamR-10M': lambda: md.Stream(md.StreamConfig(            n_embd=256, n_layer=16, n_retrieval=2, ssm_d_state=16, n_predict=4,            block_size=T, dropout=0.0, bias=False, n_attn_head=4,            window_size=128, n_global=16)).to(device),        'GPT-8L': lambda: GPTByte(n_embd=256, n_layer=8, n_head=4, n_predict=4).to(device),    }    for name, fn in builders.items():        if name in skip:            print(f'{name}: skipped at this T (see note)', flush=True)            continue        torch.manual_seed(0)        m = fn()        if USE_TRITON and _HAS_TS and isinstance(m, md.Stream):            _ts.enable_triton(m, auto=True)   # shape-conditional fused/chunked scan        n = sum(p.numel() for p in m.parameters())        print(f'{name}: {n/1e6:.3f}M params', flush=True)        evals, times = train_model(name, m, tr, va, steps, warmup, eval_at=eval_at)        models[name] = dict(evals=evals, times=times, params=n)        print(f'  -> done {name}: per-step median {med(times)*1000:.1f}ms | '              f'final val {evals[-1][1]:.4f} @ step {evals[-1][0]}', flush=True)    print()    for name, d in models.items():        print(f'  {name:12s} val: ' + ' | '.join(f'step {it} = {vl:.4f}' for it, vl in d['evals']))    for name, d in models.items():        print(f'  {name:12s} step median: {med(d["times"])*1000:.1f} ms')    return models# --- run both legs ---# GPT is skipped at T=16384 on purpose: its O(n^2) attention does not fit a T4# at that length (scores alone ~2GB fp16), which is exactly the gap Stream# exists to close. Long-context leg = Stream vs StreamR.leg1 = run_leg(4096, 1200, 100, B=4, tag='A quality')leg2 = run_leg(16384, 400, 50, B=2, tag='B long-context', skip=('GPT-8L',))# --- summary + verdict ---def row(leg, name):    d = leg.get(name)    return (d['evals'][-1][1], med(d['times'])) if d else (float('nan'), float('nan'))print(f'\n{"="*70}', flush=True)print('SUMMARY (final val loss / per-step median ms)', flush=True)print(f'{"="*70}', flush=True)for tag, leg in [('T=4096', leg1), ('T=16384', leg2)]:    sl, st = row(leg, 'Stream-10M')    rl, rt = row(leg, 'StreamR-10M')    gl, gt = row(leg, 'GPT-8L')    print(f'  {tag}:')    print(f'    val loss : Stream {sl:.4f} | StreamR {rl:.4f} | GPT {gl:.4f}')    print(f'    step ms  : Stream {st*1000:.1f} | StreamR {rt*1000:.1f} | GPT {gt*1000:.1f}')    if tag == 'T=4096':        print(f'    retrieval helps (StreamR < Stream): {rl < sl} | '              f'gap-to-GPT closed: {(sl-gl)-(rl-gl):.4f} | StreamR faster than GPT: {rt < gt}')    else:        print(f'    retrieval helps (StreamR < Stream): {rl < sl} | '              f'    GPT did NOT fit T=16384 on T4 (O(n^2) attention) - the gap this model is built to close.')